# Governing the AI Economic Transition
## Parametric Model — Interactive Companion

**Companion to the policy brief — March 2026**

*This notebook lets you reproduce every chart from the paper and run your own scenarios. Change any parameter and see how the structural relationships hold — or break. All assumptions are explicit; nothing is hidden.*

**Eight parameters control the model:**
| Parameter | What it controls |
|---|---|
| `mid` | Years to 50% knowledge-work automation (5=fast, 9=medium, 14=slow) |
| `yeomen` | Fraction of AI capital income to small owner-operators (0–0.60) |
| `dao_frac` | Fraction of knowledge-work income through commons DAOs (0–0.25) |
| `levy_prog` | Compute levy progressivity (0=flat, 1=highly progressive) |
| `dao_govt_rate` | Rate of govt asset acquisition and daoification (0–0.20) |
| `public_ai_frac` | Fraction of AI/robot capital as public infrastructure (0–0.90) |
| `tax_k` | Statutory capital income tax rate (0.20–0.50) |
| `enforcement` | Fraction of `tax_k` actually collectible (0.30=small nation, 1.0=large bloc) |

**Key sensitivities flagged in the paper:**
- Ownership structure outperforms redistribution — driven primarily by the MPC differential. See the *MPC Sensitivity* section to stress-test this.
- Governance decay compounds concentrated-ownership scenarios over time. Enable `enable_decay=True` in `run()` to see the endogenous doom loop.
- The yeoman scenario requires a supported equilibrium (subsidised financing + surtax backstop), not just market fragmentation.


## Setup

In [ ]:
# Install dependencies (only needed on Colab)
import subprocess, sys
subprocess.check_call([sys.executable, '-m', 'pip', 'install', 'numpy', 'pandas', 'matplotlib', '-q'])
print('Dependencies ready.')

## Model Constants

All parameters are explicit and adjustable. The model is calibrated to the US in 2026.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import warnings
warnings.filterwarnings('ignore')

# ── Economy baseline ─────────────────────────────────────────────────────────
GDP_0       = 28_000   # $bn nominal GDP, 2026
LABOR_FORCE = 160.0    # millions of workers

# Sector shares — BEA Value Added by Industry, 2024
# Source: U.S. Bureau of Economic Analysis, GDP by Industry; Statista/BEA (2025)
# https://www.bea.gov/itable/gdp-by-industry
KNOW_GDP_SHARE        = 0.28   # finance/insurance + professional services + information + education
PHYS_GDP_SHARE        = 0.25   # manufacturing + construction + transport + wholesale + mining/util/ag
SVCS_GDP_SHARE        = 0.11   # retail trade + accommodation/food + personal services (automatable)
GOVT_GDP_SHARE        = 0.11   # government VALUE-ADDED only (not total outlays; BEA 2024: 11.3%)
REAL_ESTATE_GDP_SHARE = 0.13   # mostly imputed rent — treated as capital income, not automatable production
# Residual (~0.12) is the baseline human economy (personal care, arts, bespoke services)

# Worker shares — BLS Employment & Wages 2024
# Source: U.S. Bureau of Labor Statistics, Employment by Major Industry Sector
# https://www.bls.gov/emp/tables/employment-by-major-industry-sector.htm
KNOW_WORKER_SHARE  = 0.22   # ~35M: professional/business services, finance, information, education
PHYS_WORKER_SHARE  = 0.24   # ~38M: manufacturing, construction, transport, wholesale, mining/ag
SVCS_WORKER_SHARE  = 0.22   # ~35M: retail, food service, accommodation, personal services

# ── Automation ceilings and price dynamics ───────────────────────────────────
KNOW_AUTO_CEILING  = 0.88
PHYS_AUTO_CEILING  = 0.68
SVCS_AUTO_CEILING  = 0.75   # higher than physical (software-only); lower than knowledge (human touch floor)
PHYS_LAG_YEARS     = 7      # physical automation lags knowledge (robot hardware deployment)
SVCS_LAG_YEARS     = 3      # services lag knowledge but faster than physical (software-driven)

# Knowledge sector
KNOW_DEFL_INITIAL  = 0.10   # deflation rate at t=0; falls asymptotically to terminal
KNOW_DEFL_TERMINAL = 0.02
KNOW_DEFL_HALFLIFE = 8
KNOW_BASE_GROWTH   = 0.02   # baseline demand growth: more people/businesses buy knowledge services
                             # as they get cheaper (Jevons effect on quantity demanded)
KNOW_REAL_GROWTH   = 0.06   # additional real output expansion from AI capability gains

# Physical sector
PHYS_DEFL_INITIAL  = 0.03
PHYS_DEFL_TERMINAL = 0.01
PHYS_BASE_GROWTH   = 0.015
PHYS_REAL_GROWTH   = 0.04

# Services sector (automatable: retail, food, hospitality, personal services)
SVCS_DEFL_INITIAL  = 0.05   # high labour content → steep price drop when automated
SVCS_DEFL_TERMINAL = 0.01
SVCS_BASE_GROWTH   = 0.015  # population + income-driven demand
SVCS_REAL_GROWTH   = 0.03   # volume gains from automation (more throughput per outlet)

# Services → human economy transition
SVCS_TO_HUMAN_FRAC = 0.40   # fraction of displaced services workers who pivot to genuine human
                             # economy work — policy lever, higher with retraining programmes

# Real estate
REAL_ESTATE_GROWTH    = 0.025  # baseline: population + housing appreciation
RE_AUTO_UPLIFT        = 0.025  # additional growth per unit of avg automation:
                               # AI-enabled development (precision construction,
                               # logistics optimisation, precision ag) makes
                               # previously marginal land valuable — expands the
                               # supply of "prime" land rather than reducing scarcity

HUMAN_PRICE_INFL   = 0.04   # Baumol cost disease — human economy wages rise with productivity

# ── Government ───────────────────────────────────────────────────────────────
# NOTE: EXISTING_GOVT_BASE_PCT is government OUTLAYS (purchases + operations, excl. transfers)
# as % of GDP — distinct from GOVT_GDP_SHARE (value-added only).
EXISTING_GOVT_BASE_PCT = 0.215
DEBT_INITIAL_BN        = 34_000
DEBT_INTEREST_RATE     = 0.045
TAX_LABOR_RATE    = 0.25
TAX_VAT_RATE      = 0.12

# ── Capital ──────────────────────────────────────────────────────────────────
CAPITAL_GDP_RATIO_0 = 3.5
CAPITAL_ACCUM_RATE  = 0.04

CAP_SHARE_TOP1   = 0.38
CAP_SHARE_NEXT9  = 0.51
CAP_SHARE_PENSION= 0.10
CAP_SHARE_BOT50  = 0.01

MPC_BASE_TOP1    = 0.12
MPC_BASE_NEXT9   = 0.30
MPC_BASE_PENSION = 0.72
MPC_BASE_BOT50   = 0.90

HUMAN_SVC_SHARE_CAPITAL = 0.15
HUMAN_SVC_SHARE_UBI     = 0.06
HUMAN_SVC_SHARE_LABOR   = 0.10

MPC_YEOMEN = 0.78
MPC_DAO    = 0.78
HUMAN_SVC_SHARE_DAO = 0.12

# ── 2026 baseline values (all scenarios start here) ──────────────────────────
YEOMEN_BASE       = 0.08
TAX_K_BASE        = 0.21
DAO_BASE          = 0.00
POLICY_RAMP_YEARS = 10

# ── Home production buffer ───────────────────────────────────────────────────
LAND_ACCESS_FRAC     = 0.40
HOME_PROD_VALUE_K    = 6.0
SUBSISTENCE_COST_K   = 16.0
HOME_PROD_UBI_UPLIFT = 0.08

# ── Capital returns ──────────────────────────────────────────────────────────
R_AI_INITIAL  = 0.28
R_AI_TERMINAL = 0.07
R_AI_HALFLIFE = 11
R_YEOMEN      = 0.08
R_LAND_YIELD  = 0.035
R_LAND_APPREC = 0.035
R_ENERGY_0    = 0.07
R_ENERGY_GROW = 0.012

# ── Government daoification ──────────────────────────────────────────────────
DAOGOV_ACQUISITION_COST = 0.15

# ── Public AI infrastructure ─────────────────────────────────────────────────
COMPUTE_DIVIDEND_FRAC = 0.70
COMPUTE_SUBSIDY_FRAC  = 0.20
COMPUTE_OVERHEAD_FRAC = 0.10
ADULT_POPULATION      = 260.0
PUBLIC_AI_BASE        = 0.00
PUBLIC_AI_DECAY       = 0.033  # institutional drag from operating state infrastructure:
                               # bureaucratic capture, political pricing interference,
                               # underinvestment incentives. Calibrated to G≈0.75 at t+30
                               # for public AI 90% — worse than distributed ownership,
                               # better than private concentrated ownership.

print('Constants loaded.')
_residual = 1 - KNOW_GDP_SHARE - PHYS_GDP_SHARE - SVCS_GDP_SHARE - GOVT_GDP_SHARE - REAL_ESTATE_GDP_SHARE
print(f'Sector shares: KNOW={KNOW_GDP_SHARE} PHYS={PHYS_GDP_SHARE} SVCS={SVCS_GDP_SHARE} '
      f'GOVT={GOVT_GDP_SHARE} RE={REAL_ESTATE_GDP_SHARE} → residual (human baseline)={_residual:.2f}')

## Helper Functions

In [ ]:
def s_curve(t, ceiling, speed, midpoint):
    """
    Logistic S-curve normalised so t=0 always returns 0.
    All scenarios start from the same 2026 baseline.
    """
    raw  = ceiling / (1.0 + np.exp(-speed * (t - midpoint)))
    init = ceiling / (1.0 + np.exp(-speed * (0 - midpoint)))
    result = (raw - init) * (ceiling / (ceiling - init))
    return np.clip(result, 0.0, ceiling)


def mpc_adjusted(base_mpc, labour_income_decline_frac, sensitivity=0.5):
    """
    MPC rises toward 0.90 as labour income falls.
    Workers who lose wages and must live on capital/dividend income
    spend a higher fraction of that income.
    """
    target = 0.90
    adjustment = (target - base_mpc) * labour_income_decline_frac * sensitivity
    return np.minimum(base_mpc + adjustment, target)


print('Helpers loaded.')

## Core Model

The `run()` function takes scenario parameters and returns a 35-column annual DataFrame for 2026–2061.

In [ ]:
def run(mid, yeomen, tax_k, dao_frac=0.0, enforcement=1.0,
        levy_prog=0.0, dao_govt_rate=0.0, public_ai_frac=0.0,
        n_years=35, base_year=2026, enable_decay=False):
    """
    Run one scenario.

    Sector taxonomy (BEA 2024)
    --------------------------
    Knowledge  (28%): finance/ins, professional services, information, education — automatable
    Physical   (25%): manufacturing, construction, transport, wholesale, mining/util/ag — automatable
    Services   (11%): retail, food/hospitality, personal services — automatable (software-led)
    Government (11%): government value-added (not total outlays)
    Real estate(13%): mostly imputed rent — capital income, not automatable production
    Human econ (~12% baseline + demand-driven growth):
        - Seeded from GDP residual (existing therapists, musicians, carers etc.)
        - Grows via (a) redistribution demand, (b) services workers who pivot in
    """
    t    = np.arange(n_years, dtype=float)
    year = base_year + t

    # ── Governance decay ────────────────────────────────────────────────────
    if enable_decay:
        G      = np.zeros(n_years);  G[0] = 1.0
        _yr    = YEOMEN_BASE + (yeomen   - YEOMEN_BASE) * np.minimum(np.arange(n_years) / POLICY_RAMP_YEARS, 1.0)
        _dr    = DAO_BASE    + (dao_frac - DAO_BASE)    * np.minimum(np.arange(n_years) / POLICY_RAMP_YEARS, 1.0)
        _pai   = PUBLIC_AI_BASE + (public_ai_frac  - PUBLIC_AI_BASE) * np.minimum(np.arange(n_years) / POLICY_RAMP_YEARS, 1.0)
        for i in range(1, n_years):
            # Pressure from private concentrated capital only; public AI reduces this
            private_conc_i = (1 - _yr[i]) * (1 - _dr[i]) * (1 - _pai[i])
            pressure_i     = private_conc_i * 0.7
            # Recovery: mean-reversion toward 1.0 when below target (α=0.02)
            # + active strengthening when ownership is distributed (γ=0.03).
            # γ is set so that at max-yeomen pressure (0.28), recovery ≈ decay,
            # keeping G stable near 1.0. Concentrated pressure (0.7) still
            # overwhelms recovery and G decays rapidly.
            recovery_i = 0.02 * max(1.0 - G[i-1], 0) + 0.03 * (1.0 - pressure_i) * G[i-1]
            # Public AI adds its own drag: bureaucratic capture, political pricing
            public_drag_i  = PUBLIC_AI_DECAY * _pai[i] * G[i-1]
            G[i] = min(G[i-1] + recovery_i - 0.08 * G[i-1] * pressure_i - public_drag_i, 1.0)
            G[i] = max(G[i], 0.10)
        enforcement_eff = enforcement * G
    else:
        enforcement_eff = np.full(n_years, enforcement)
        G               = np.ones(n_years)

    # ── Policy ramp-in ──────────────────────────────────────────────────────
    ramp        = np.minimum(t / POLICY_RAMP_YEARS, 1.0)
    yeomen_t    = YEOMEN_BASE    + (yeomen         - YEOMEN_BASE)    * ramp
    dao_frac_t  = DAO_BASE       + (dao_frac        - DAO_BASE)       * ramp
    tax_k_t     = TAX_K_BASE     + (tax_k           - TAX_K_BASE)     * ramp
    public_ai_t = PUBLIC_AI_BASE + (public_ai_frac  - PUBLIC_AI_BASE) * ramp

    # ── Automation S-curves ─────────────────────────────────────────────────
    know_auto = s_curve(t, KNOW_AUTO_CEILING, 0.50, mid)
    phys_auto = s_curve(t, PHYS_AUTO_CEILING, 0.28, mid + PHYS_LAG_YEARS)
    svcs_auto = s_curve(t, SVCS_AUTO_CEILING, 0.35, mid + SVCS_LAG_YEARS)

    # ── Worker displacement ─────────────────────────────────────────────────
    know_displaced  = LABOR_FORCE * KNOW_WORKER_SHARE * know_auto
    phys_displaced  = LABOR_FORCE * PHYS_WORKER_SHARE * phys_auto
    svcs_displaced  = LABOR_FORCE * SVCS_WORKER_SHARE * svcs_auto

    # Services → human economy pivot: these workers earn wages (not UBI)
    svcs_to_human   = svcs_displaced * SVCS_TO_HUMAN_FRAC
    total_displaced = know_displaced + phys_displaced + svcs_displaced

    # ── Nominal GDP — sector by sector ──────────────────────────────────────

    # Knowledge: asymptotic deflation + base demand growth (Jevons quantity effect)
    # + real output expansion (AI capability gains). Loop mirrors physical/services.
    know_defl_t = KNOW_DEFL_TERMINAL + (KNOW_DEFL_INITIAL - KNOW_DEFL_TERMINAL) * 2**(-t / KNOW_DEFL_HALFLIFE)
    know_price  = np.ones(n_years)
    know_nom    = np.empty(n_years);  know_nom[0] = GDP_0 * KNOW_GDP_SHARE
    for i in range(1, n_years):
        know_price[i] = know_price[i-1] * (1 - know_defl_t[i])
        know_nom[i]   = know_nom[i-1]   * (1 + KNOW_BASE_GROWTH + KNOW_REAL_GROWTH) * (1 - know_defl_t[i])

    # Physical: base demand + automation real growth, asymptotic deflation
    phys_defl_t = PHYS_DEFL_TERMINAL + (PHYS_DEFL_INITIAL - PHYS_DEFL_TERMINAL) * phys_auto
    phys_real_t = PHYS_REAL_GROWTH * phys_auto
    phys_price  = np.ones(n_years)
    phys_nom    = np.empty(n_years);  phys_nom[0] = GDP_0 * PHYS_GDP_SHARE
    for i in range(1, n_years):
        phys_price[i] = phys_price[i-1] * (1 - phys_defl_t[i])
        phys_nom[i]   = phys_nom[i-1]   * (1 + PHYS_BASE_GROWTH + phys_real_t[i]) * (1 - phys_defl_t[i])

    # Services: same structure as physical
    svcs_defl_t = SVCS_DEFL_TERMINAL + (SVCS_DEFL_INITIAL - SVCS_DEFL_TERMINAL) * svcs_auto
    svcs_real_t = SVCS_REAL_GROWTH * svcs_auto
    svcs_price  = np.ones(n_years)
    svcs_nom    = np.empty(n_years);  svcs_nom[0] = GDP_0 * SVCS_GDP_SHARE
    for i in range(1, n_years):
        svcs_price[i] = svcs_price[i-1] * (1 - svcs_defl_t[i])
        svcs_nom[i]   = svcs_nom[i-1]   * (1 + SVCS_BASE_GROWTH + svcs_real_t[i]) * (1 - svcs_defl_t[i])

    # Government value-added (not outlays)
    govt_nom = GDP_0 * GOVT_GDP_SHARE * (1.02 ** t)

    # Real estate: grows at baseline + automation uplift.
    # Automation expands the effective supply of valuable land (precision construction,
    # better logistics, smart-grid infrastructure) so more marginal land enters the
    # premium market even as per-unit scarcity rents persist.
    avg_auto = (know_auto + phys_auto + svcs_auto) / 3.0
    re_growth_t = REAL_ESTATE_GROWTH + RE_AUTO_UPLIFT * avg_auto
    real_estate_nom = np.empty(n_years); real_estate_nom[0] = GDP_0 * REAL_ESTATE_GDP_SHARE
    for i in range(1, n_years):
        real_estate_nom[i] = real_estate_nom[i-1] * (1 + re_growth_t[i])

    # Human economy baseline: existing activity (therapists, musicians, carers etc.)
    # present at t=0 BEFORE any AI-driven redistribution. Seeded from GDP residual;
    # grows at 2%/yr nominal (Baumol wages up, volumes roughly flat pre-AI boom).
    _human_share_0  = 1 - KNOW_GDP_SHARE - PHYS_GDP_SHARE - SVCS_GDP_SHARE - GOVT_GDP_SHARE - REAL_ESTATE_GDP_SHARE
    human_baseline  = GDP_0 * _human_share_0 * (1.02 ** t)

    # ── Income from automatable sectors ────────────────────────────────────
    know_labor        = know_nom * (1 - know_auto) * 0.60
    phys_labor        = phys_nom * (1 - phys_auto) * 0.58
    svcs_labor        = svcs_nom * (1 - svcs_auto) * 0.65
    auto_labor_income = know_labor + phys_labor + svcs_labor

    know_gross_cap    = know_nom * (know_auto * 0.60 + 0.40)
    phys_gross_cap    = phys_nom * (phys_auto * 0.58 + 0.42)
    svcs_gross_cap    = svcs_nom * (svcs_auto * 0.65 + 0.35)
    re_cap_income     = real_estate_nom * 0.85

    gross_capital_income = know_gross_cap + phys_gross_cap + svcs_gross_cap + re_cap_income

    # ── Public AI carve-out ─────────────────────────────────────────────────
    compute_pool_income     = gross_capital_income * public_ai_t
    private_cap_income      = gross_capital_income * (1 - public_ai_t)
    compute_dividend_total  = compute_pool_income * COMPUTE_DIVIDEND_FRAC
    compute_subsidy_total   = compute_pool_income * COMPUTE_SUBSIDY_FRAC
    compute_overhead        = compute_pool_income * COMPUTE_OVERHEAD_FRAC
    compute_dividend_k      = compute_dividend_total / ADULT_POPULATION

    # ── Ownership structure ─────────────────────────────────────────────────
    levy_yeomen_bonus = levy_prog * 0.25 * ramp
    yeomen_effective  = np.minimum(yeomen_t + levy_yeomen_bonus, 0.85)

    yeomen_income   = private_cap_income * yeomen_effective
    dao_income      = know_gross_cap * (1 - public_ai_t) * (1 - yeomen_effective) * dao_frac_t
    conc_cap_income = (know_gross_cap * (1 - public_ai_t) * (1 - yeomen_effective) * (1 - dao_frac_t)
                     + phys_gross_cap * (1 - public_ai_t) * (1 - yeomen_effective)
                     + svcs_gross_cap * (1 - public_ai_t) * (1 - yeomen_effective)
                     + re_cap_income  * (1 - public_ai_t) * (1 - yeomen_effective))

    displacement_frac = (know_auto + phys_auto + svcs_auto) / 3.0
    govt_dao_income   = dao_govt_rate * conc_cap_income * displacement_frac
    govt_dao_cost     = DAOGOV_ACQUISITION_COST * govt_dao_income
    conc_cap_income   = conc_cap_income - govt_dao_income

    # ── MPC adjustment ──────────────────────────────────────────────────────
    labor_income_0  = GDP_0 * (KNOW_GDP_SHARE * 0.60 + PHYS_GDP_SHARE * 0.58 + SVCS_GDP_SHARE * 0.65)
    labour_decline  = np.clip((labor_income_0 - auto_labor_income) / labor_income_0, 0, 1)

    mpc_top1    = mpc_adjusted(MPC_BASE_TOP1,    labour_decline, 0.30)
    mpc_next9   = mpc_adjusted(MPC_BASE_NEXT9,   labour_decline, 0.55)
    mpc_pension = mpc_adjusted(MPC_BASE_PENSION, labour_decline, 0.20)
    mpc_bot50   = mpc_adjusted(MPC_BASE_BOT50,   labour_decline, 0.05)

    mpc_conc = (CAP_SHARE_TOP1    * mpc_top1   +
                CAP_SHARE_NEXT9   * mpc_next9  +
                CAP_SHARE_PENSION * mpc_pension +
                CAP_SHARE_BOT50   * mpc_bot50)

    # ── Consumption ─────────────────────────────────────────────────────────
    conc_consumption        = conc_cap_income       * mpc_conc
    yeomen_conc             = yeomen_income          * MPC_YEOMEN
    dao_consumption         = dao_income             * MPC_DAO
    govt_dao_consumption    = govt_dao_income        * MPC_DAO
    compute_div_consumption = compute_dividend_total * 0.85
    labour_conc             = auto_labor_income      * 0.90

    # ── Government finances — pass 1 ────────────────────────────────────────
    wealth_tax_rate    = 0.01 if tax_k >= 0.40 else 0.005
    K                  = GDP_0 * CAPITAL_GDP_RATIO_0 * (1 + CAPITAL_ACCUM_RATE) ** t
    interest_burden_bn = np.full(n_years, DEBT_INITIAL_BN * DEBT_INTEREST_RATE)
    tax_k_effective    = tax_k_t * enforcement_eff
    levy_multiplier    = 1.0 + levy_prog * 0.50
    tax_conc_rate      = tax_k_effective * levy_multiplier

    tax_conc   = tax_conc_rate * conc_cap_income
    tax_yeomen = TAX_LABOR_RATE * yeomen_income
    tax_dao    = TAX_LABOR_RATE * dao_income
    tax_labor  = TAX_LABOR_RATE * auto_labor_income
    tax_wealth = wealth_tax_rate * K

    gdp_auto_govt   = know_nom + phys_nom + svcs_nom + govt_nom + real_estate_nom
    tax_vat_pass1   = TAX_VAT_RATE * gdp_auto_govt * 0.65
    tax_total_pass1 = tax_conc + tax_yeomen + tax_dao + tax_labor + tax_wealth + tax_vat_pass1
    base_spending_t = GDP_0 * EXISTING_GOVT_BASE_PCT * (0.40 + 0.60 * (gdp_auto_govt / GDP_0))
    existing_spending     = base_spending_t + interest_burden_bn + govt_dao_cost + compute_overhead
    redistributable_pass1 = np.maximum(tax_total_pass1 - existing_spending, 0.0)

    ubi_pass1          = np.where(total_displaced > 0.5, redistributable_pass1 / total_displaced, 0.0)
    ubi_pass1          = np.minimum(ubi_pass1, 60.0)
    ubi_total_spending = ubi_pass1 * total_displaced * 0.85

    # ── Human economy ────────────────────────────────────────────────────────
    # Demand-driven component (redistribution spillover)
    human_demand = (
        conc_consumption        * HUMAN_SVC_SHARE_CAPITAL +
        yeomen_conc             * HUMAN_SVC_SHARE_LABOR   +
        dao_consumption         * HUMAN_SVC_SHARE_DAO     +
        govt_dao_consumption    * HUMAN_SVC_SHARE_DAO     +
        compute_div_consumption * HUMAN_SVC_SHARE_UBI     +
        compute_subsidy_total   * HUMAN_SVC_SHARE_LABOR   +
        labour_conc             * HUMAN_SVC_SHARE_LABOR   +
        ubi_total_spending      * HUMAN_SVC_SHARE_UBI
    )
    # Services pivot workers earn wages at wage floor, generating additional human demand
    human_wage_floor       = 60   # $k/yr
    svcs_human_wage_income = svcs_to_human * human_wage_floor   # $bn
    human_demand          += svcs_human_wage_income * 0.85 * HUMAN_SVC_SHARE_LABOR

    # Total human economy = existing baseline stock + demand-driven increment
    # Dynamic Keynesian multiplier: 1/(1 - MPC_worker × human_share)
    # Human share is derived from this year's pre-multiplier outputs — no circularity.
    # As human economy grows relative to automated sectors, the multiplier rises,
    # creating scenario differentiation: yeomen ~1.15+, concentrated ~1.07.
    _gdp_auto_only     = know_nom + phys_nom + svcs_nom + govt_nom + real_estate_nom
    _human_pre         = human_baseline + human_demand
    _human_share_t     = _human_pre / np.maximum(_human_pre + _gdp_auto_only, 1.0)
    _human_worker_mpc  = 0.85
    human_multiplier   = 1.0 / np.maximum(1.0 - _human_worker_mpc * _human_share_t, 0.3)
    human_nom          = _human_pre * human_multiplier
    human_labor_income = human_nom * 0.82
    # Existing baseline workers already hold pre-AI human economy jobs.
    # Displaced workers can only fill NEW positions above the t=0 headcount.
    human_baseline_workers = (GDP_0 * _human_share_0 * 0.82) / human_wage_floor
    human_workers_max  = human_labor_income / human_wage_floor
    human_new_jobs     = np.maximum(human_workers_max - human_baseline_workers, 0.0)
    human_workers      = np.maximum(human_new_jobs + svcs_to_human, 0.1)
    net_without_income = np.maximum(total_displaced - human_workers, 0.0)

    # ── Full GDP ─────────────────────────────────────────────────────────────
    gdp_nom = know_nom + phys_nom + svcs_nom + human_nom + govt_nom + real_estate_nom

    # ── Refined government finances ──────────────────────────────────────────
    tax_vat        = TAX_VAT_RATE * gdp_nom * 0.65
    tax_total      = tax_conc + tax_yeomen + tax_dao + tax_labor + tax_wealth + tax_vat
    base_spending_t   = GDP_0 * EXISTING_GOVT_BASE_PCT * (0.40 + 0.60 * (gdp_nom / GDP_0))
    existing_spending = base_spending_t + interest_burden_bn + govt_dao_cost + compute_overhead
    redistributable   = np.maximum(tax_total - existing_spending, 0.0)

    ubi_per_person = np.where(net_without_income > 0.5, redistributable / net_without_income, 0.0)
    ubi_per_person = np.minimum(ubi_per_person, 60.0)

    # ── Home production buffer ───────────────────────────────────────────────
    home_prod_avg       = HOME_PROD_VALUE_K * LAND_ACCESS_FRAC
    effective_welfare_k = ubi_per_person + compute_dividend_k + home_prod_avg

    ubi_land_access    = ubi_per_person * net_without_income * LAND_ACCESS_FRAC
    extra_human_demand = ubi_land_access * HOME_PROD_UBI_UPLIFT
    human_nom          = human_nom + extra_human_demand
    human_labor_income = human_nom * 0.82
    human_workers_max  = human_labor_income / human_wage_floor
    human_new_jobs     = np.maximum(human_workers_max - human_baseline_workers, 0.0)
    human_workers      = np.maximum(human_new_jobs + svcs_to_human, 0.1)
    human_wage_k       = human_labor_income / np.maximum(human_workers - svcs_to_human, 0.1)
    net_without_income = np.maximum(total_displaced - human_workers, 0.0)

    # ── Income distribution ──────────────────────────────────────────────────
    total_labor_income   = auto_labor_income + human_labor_income + yeomen_income + dao_income + govt_dao_income
    total_capital_income = conc_cap_income
    total_income         = total_labor_income + total_capital_income + compute_dividend_total
    labor_share_pct      = total_labor_income / total_income * 100

    capital_share_frac    = total_capital_income / total_income
    gini_proxy = 0.35 * (1 - capital_share_frac) + 0.90 * capital_share_frac * (1 - yeomen_t)
    gini_proxy += 0.45 * capital_share_frac * yeomen_t
    dividend_income_share = compute_dividend_total / np.maximum(total_income, 1)
    gini_proxy            = gini_proxy * (1 - dividend_income_share)

    # ── Real GDP index ───────────────────────────────────────────────────────
    w_k = KNOW_GDP_SHARE; w_p = PHYS_GDP_SHARE; w_s = SVCS_GDP_SHARE; w_h = 0.05
    human_price     = (1 + HUMAN_PRICE_INFL) ** t
    composite_price = (w_k * know_price + w_p * phys_price + w_s * svcs_price + w_h * human_price) / (w_k + w_p + w_s + w_h)
    gdp_real_idx    = (gdp_nom / composite_price) / GDP_0 * 100

    # ── Capital returns ──────────────────────────────────────────────────────
    r_ai     = (R_AI_INITIAL - R_AI_TERMINAL) * 2**(-t / R_AI_HALFLIFE) + R_AI_TERMINAL
    r_yeomen = R_YEOMEN * np.ones(n_years)
    r_land   = (R_LAND_YIELD + R_LAND_APPREC) * np.ones(n_years)
    r_energy = np.minimum(R_ENERGY_0 + R_ENERGY_GROW * t, 0.18)

    distributed_frac = yeomen_t + (1 - yeomen_t) * dao_frac_t
    w_ai     = (1 - distributed_frac) * 0.50 * 2**(-t / 10)
    w_yeo    = distributed_frac        * 0.50 * (1 - 0.3 * 2**(-t / 10))
    w_land   = 0.25;  w_energy = 0.15
    w_other  = np.maximum(1 - w_ai - w_yeo - w_land - w_energy, 0.05)
    total_w  = w_ai + w_yeo + w_land + w_energy + w_other
    r_blended = (w_ai * r_ai + w_yeo * r_yeomen + w_land * r_land +
                 w_energy * r_energy + w_other * 0.04) / total_w

    mpc_aggregate = (conc_cap_income * mpc_conc + yeomen_income * MPC_YEOMEN +
                     dao_income * MPC_DAO + auto_labor_income * 0.90) / np.maximum(total_income, 1)

    # ── Fiscal analysis ──────────────────────────────────────────────────────
    tax_lost_to_leakage  = tax_k * (1 - enforcement) * conc_cap_income
    fiscal_space_bn      = tax_total - existing_spending
    ubi_total_final      = ubi_per_person * net_without_income
    true_surplus         = fiscal_space_bn - ubi_total_final
    debt                 = np.zeros(n_years);  debt[0] = DEBT_INITIAL_BN
    for i in range(1, n_years):
        debt[i] = max(debt[i-1] - true_surplus[i], 0.0)
    debt_gdp_ratio       = debt / gdp_nom
    interest_pct_revenue = interest_burden_bn / np.maximum(tax_total, 1) * 100

    return pd.DataFrame({
        'year': year, 't': t,
        'gdp_nom_bn': gdp_nom, 'gdp_real_idx': gdp_real_idx,
        'know_nom': know_nom, 'phys_nom': phys_nom, 'svcs_nom': svcs_nom,
        'govt_nom': govt_nom, 'real_estate_nom': real_estate_nom,
        'human_nom_bn': human_nom, 'human_pct_gdp': human_nom / gdp_nom * 100,
        'total_displaced_m': total_displaced, 'human_workers_m': human_workers,
        'svcs_to_human_m': svcs_to_human, 'net_without_income_m': net_without_income,
        'know_auto_pct': know_auto * 100, 'phys_auto_pct': phys_auto * 100, 'svcs_auto_pct': svcs_auto * 100,
        'labor_income_bn': total_labor_income, 'capital_income_bn': total_capital_income,
        'yeomen_income_bn': yeomen_income, 'dao_income_bn': dao_income,
        'govt_dao_income_bn': govt_dao_income,
        'compute_pool_bn': compute_pool_income, 'compute_dividend_k': compute_dividend_k,
        'yeomen_effective': yeomen_effective,
        'labor_share_pct': labor_share_pct, 'gini_proxy': gini_proxy,
        'mpc_aggregate': mpc_aggregate * 100,
        'tax_total_bn': tax_total, 'tax_k_effective': tax_k_effective,
        'governance_quality': G,
        'human_multiplier': human_multiplier,
        'tax_lost_leakage_bn': tax_lost_to_leakage,
        'redistributable_bn': redistributable, 'ubi_per_person_k': ubi_per_person,
        'existing_spending_bn': existing_spending, 'interest_burden_bn': interest_burden_bn,
        'debt_gdp_ratio': debt_gdp_ratio, 'interest_pct_revenue': interest_pct_revenue,
        'fiscal_space_bn': fiscal_space_bn,
        'effective_welfare_k': effective_welfare_k,
        'human_wage_k': human_wage_k,
        'r_ai_pct': r_ai * 100, 'r_yeomen_pct': r_yeomen * 100,
        'r_land_pct': r_land * 100, 'r_energy_pct': r_energy * 100,
        'r_blended_pct': r_blended * 100,
    })

print('Model loaded.')

## Pre-defined Scenarios

These are the scenarios from the policy brief. You can add your own below.

In [ ]:
SCENARIOS = {
    # Baseline comparisons
    'Fast / No yeomen / Low tax':         dict(mid=5,  yeomen=0.00, dao_frac=0.00, tax_k=0.20, enforcement=1.00, levy_prog=0.0, dao_govt_rate=0.00, public_ai_frac=0.00),
    'Fast / No yeomen / High tax':        dict(mid=5,  yeomen=0.00, dao_frac=0.00, tax_k=0.50, enforcement=1.00, levy_prog=0.0, dao_govt_rate=0.00, public_ai_frac=0.00),
    'Fast / Moderate yeomen / Med tax':   dict(mid=5,  yeomen=0.35, dao_frac=0.00, tax_k=0.35, enforcement=1.00, levy_prog=0.0, dao_govt_rate=0.00, public_ai_frac=0.00),
    'Fast / High yeomen / High tax':      dict(mid=5,  yeomen=0.60, dao_frac=0.00, tax_k=0.50, enforcement=1.00, levy_prog=0.0, dao_govt_rate=0.00, public_ai_frac=0.00),
    'Medium / Moderate yeomen / Med tax': dict(mid=9,  yeomen=0.35, dao_frac=0.00, tax_k=0.35, enforcement=1.00, levy_prog=0.0, dao_govt_rate=0.00, public_ai_frac=0.00),
    'Slow / Moderate yeomen / Med tax':   dict(mid=14, yeomen=0.35, dao_frac=0.00, tax_k=0.35, enforcement=1.00, levy_prog=0.0, dao_govt_rate=0.00, public_ai_frac=0.00),
    'Slow / No yeomen / Low tax':         dict(mid=14, yeomen=0.00, dao_frac=0.00, tax_k=0.20, enforcement=1.00, levy_prog=0.0, dao_govt_rate=0.00, public_ai_frac=0.00),
    # DAO / commons
    'Fast / Yeomen + DAO / High tax':     dict(mid=5,  yeomen=0.35, dao_frac=0.25, tax_k=0.50, enforcement=1.00, levy_prog=0.0, dao_govt_rate=0.00, public_ai_frac=0.00),
    'Fast / Progressive levy / High tax': dict(mid=5,  yeomen=0.00, dao_frac=0.00, tax_k=0.50, enforcement=1.00, levy_prog=0.8, dao_govt_rate=0.00, public_ai_frac=0.00),
    'Fast / Govt daoify / High tax':      dict(mid=5,  yeomen=0.00, dao_frac=0.00, tax_k=0.50, enforcement=1.00, levy_prog=0.0, dao_govt_rate=0.20, public_ai_frac=0.00),
    'Fast / Full stack':                  dict(mid=5,  yeomen=0.35, dao_frac=0.20, tax_k=0.50, enforcement=1.00, levy_prog=0.8, dao_govt_rate=0.20, public_ai_frac=0.00),
    # Public AI infrastructure
    'Fast / Public AI 50%':               dict(mid=5,  yeomen=0.10, dao_frac=0.00, tax_k=0.25, enforcement=1.00, levy_prog=0.0, dao_govt_rate=0.00, public_ai_frac=0.50),
    'Fast / Public AI 90%':               dict(mid=5,  yeomen=0.10, dao_frac=0.00, tax_k=0.20, enforcement=1.00, levy_prog=0.0, dao_govt_rate=0.00, public_ai_frac=0.90),
    'Fast / Public AI + full stack':      dict(mid=5,  yeomen=0.20, dao_frac=0.10, tax_k=0.25, enforcement=1.00, levy_prog=0.3, dao_govt_rate=0.10, public_ai_frac=0.70),
    # Small nation
    'Fast / High yeomen / High tax / Small nation': dict(mid=5, yeomen=0.60, dao_frac=0.00, tax_k=0.50, enforcement=0.30, levy_prog=0.0, dao_govt_rate=0.00, public_ai_frac=0.00),
    'Fast / No yeomen / High tax / Small nation':   dict(mid=5, yeomen=0.00, dao_frac=0.00, tax_k=0.50, enforcement=0.30, levy_prog=0.0, dao_govt_rate=0.00, public_ai_frac=0.00),
    # Stress test: public AI with no yeomen (platform capture scenario)
    'Fast / Public AI 90% / No yeomen':   dict(mid=5,  yeomen=0.00, dao_frac=0.00, tax_k=0.20, enforcement=1.00, levy_prog=0.0, dao_govt_rate=0.00, public_ai_frac=0.90),
    # Governance decay stress-tests (enable_decay=True)
    'Fast / No yeomen / Low tax / Decay':    dict(mid=5, yeomen=0.00, dao_frac=0.00, tax_k=0.20, enforcement=1.00, levy_prog=0.0, dao_govt_rate=0.00, public_ai_frac=0.00, enable_decay=True),
    'Fast / No yeomen / High tax / Decay':   dict(mid=5, yeomen=0.00, dao_frac=0.00, tax_k=0.50, enforcement=1.00, levy_prog=0.0, dao_govt_rate=0.00, public_ai_frac=0.00, enable_decay=True),
    'Fast / High yeomen / High tax / Decay': dict(mid=5, yeomen=0.60, dao_frac=0.00, tax_k=0.50, enforcement=1.00, levy_prog=0.0, dao_govt_rate=0.00, public_ai_frac=0.00, enable_decay=True),
    'Fast / Full stack / Decay':             dict(mid=5, yeomen=0.35, dao_frac=0.20, tax_k=0.50, enforcement=1.00, levy_prog=0.8, dao_govt_rate=0.20, public_ai_frac=0.00, enable_decay=True),
}

# Run all scenarios
results = {}
for name, params in SCENARIOS.items():
    results[name] = run(**params)
    print(f'  ✓  {name}')

print(f'\nAll {len(results)} scenarios complete.')

## Charts

Highlighted scenarios are the key comparisons from the paper. Others shown faintly for context.

In [ ]:
COLORS = [
    '#d62728', '#ff7f0e', '#2ca02c', '#1f77b4',
    '#9467bd', '#8c564b', '#e377c2', '#17becf',
    '#bcbd22', '#aec7e8', '#f7b6d2', '#c5b0d5',
    '#c49c94', '#dbdb8d', '#9edae5', '#393b79', '#7f7f7f',
]

HIGHLIGHT = [
    'Fast / No yeomen / Low tax',
    'Fast / No yeomen / High tax',
    'Fast / High yeomen / High tax',
    'Fast / Full stack',
    'Fast / Public AI 90%',
    'Fast / Public AI + full stack',
    'Fast / Public AI 90% / No yeomen',
]

fig = plt.figure(figsize=(22, 35))
fig.suptitle(
    'AI Economic Transition Model  |  2026–2061\n'
    'Automation speed × Yeomen/DAO ownership × Capital tax rate × Enforcement',
    fontsize=13, fontweight='bold', y=0.99
)
gs = gridspec.GridSpec(5, 3, figure=fig, hspace=0.50, wspace=0.35)

panels = [
    (gs[0, 0], 'gdp_nom_bn',           '$T',       'Nominal GDP',                  1/1000),
    (gs[0, 1], 'gdp_real_idx',         'Index',    'Real GDP (2026=100)',           1),
    (gs[0, 2], 'net_without_income_m', 'Millions', 'Workers w/o Income',            1),
    (gs[1, 0], 'labor_share_pct',      '%',        'Labour Share of Income',        1),
    (gs[1, 1], 'gini_proxy',           'Gini',     'Inequality (Gini proxy)',       1),
    (gs[1, 2], 'effective_welfare_k',  '$k/yr',    'Effective Welfare (UBI+Home)',  1),
    (gs[2, 0], 'tax_total_bn',         '$T',       'Total Tax Revenue',             1/1000),
    (gs[2, 1], 'redistributable_bn',   '$T',       'Redistributable Surplus',       1/1000),
    (gs[2, 2], 'human_wage_k',         '$k/yr',    'Human Economy Wage',            1),
    (gs[3, 0], 'r_blended_pct',        '% return', 'Blended Capital Return',        1),
    (gs[3, 1], 'tax_lost_leakage_bn',  '$T',       'Tax Lost to Enforcement Gap',   1/1000),
    (gs[3, 2], 'human_pct_gdp',        '% of GDP', 'Human Economy % of GDP',       1),
    (gs[4, 0], 'debt_gdp_ratio',       'ratio',    'Debt / Nominal GDP',            1),
    (gs[4, 1], 'interest_pct_revenue', '%',        'Interest as % of Tax Revenue',  1),
    (gs[4, 2], 'fiscal_space_bn',      '$T',       'Fiscal Space (before UBI)',     1/1000),
]

axes = [fig.add_subplot(loc) for loc, *_ in panels]

for i, (name, df) in enumerate(results.items()):
    color = COLORS[i % len(COLORS)]
    hl    = name in HIGHLIGHT
    x     = df['year']
    for ax, (_, col, unit, title, scale) in zip(axes, panels):
        ax.plot(x, df[col] * scale, color=color,
                alpha=1.0 if hl else 0.22,
                linewidth=2.0 if hl else 0.8,
                label=name if hl else '_nolegend_')

for ax, (_, col, unit, title, scale) in zip(axes, panels):
    ax.set_title(title, fontsize=10, fontweight='bold')
    ax.set_ylabel(unit, fontsize=9)
    ax.set_xlabel('Year', fontsize=9)
    ax.grid(True, alpha=0.25)
    ax.set_xlim(2026, 2060)
    ax.legend(fontsize=6, loc='best', framealpha=0.7)

# Replace human_pct_gdp panel with capital returns
ax_ret = axes[11]
ax_ret.clear()
mid_df = results.get('Medium / Moderate yeomen / Med tax', list(results.values())[0])
x = mid_df['year']
ax_ret.plot(x, mid_df['r_ai_pct'],      label='Concentrated AI', lw=2, color='#d62728')
ax_ret.plot(x, mid_df['r_yeomen_pct'],  label='Yeomen',          lw=2, color='#2ca02c')
ax_ret.plot(x, mid_df['r_land_pct'],    label='Land',            lw=2, color='#ff7f0e')
ax_ret.plot(x, mid_df['r_energy_pct'],  label='Energy',          lw=2, color='#9467bd')
ax_ret.plot(x, mid_df['r_blended_pct'], label='Blended market',  lw=2, color='#1f77b4', linestyle='--')
ax_ret.set_title('Capital Returns by Asset Class', fontsize=10, fontweight='bold')
ax_ret.set_ylabel('% annual return', fontsize=9)
ax_ret.set_xlabel('Year', fontsize=9)
ax_ret.legend(fontsize=7)
ax_ret.grid(True, alpha=0.25)
ax_ret.set_xlim(2026, 2060)

plt.savefig('ai_economy.png', dpi=150, bbox_inches='tight')
plt.show()
print('Chart saved → ai_economy.png')

In [ ]:
## Governance Decay — Doom Loop Visualised
# Shows how concentrated ownership erodes institutional capacity over time,
# and how distributed ownership slows decay.
# Static model (no decay flag) vs endogenous decay (enable_decay=True)

decay_pairs = [
    ('Fast / No yeomen / Low tax',            'Fast / No yeomen / Low tax / Decay',    '#d62728', 'No yeomen / Low tax'),
    ('Fast / No yeomen / High tax',           'Fast / No yeomen / High tax / Decay',   '#ff7f0e', 'No yeomen / High tax'),
    ('Fast / High yeomen / High tax',         'Fast / High yeomen / High tax / Decay', '#2ca02c', 'High yeomen / High tax'),
    ('Fast / Full stack',                     'Fast / Full stack / Decay',             '#1f77b4', 'Full stack'),
]

fig, axes = plt.subplots(1, 3, figsize=(18, 5))
fig.suptitle('Governance Decay: Static vs Endogenous (enable_decay=True)', fontweight='bold')

for base_name, decay_name, color, label in decay_pairs:
    base_df  = results[base_name]
    decay_df = results[decay_name]
    x = base_df['year']
    axes[0].plot(x, decay_df['governance_quality'], color=color, lw=2, label=label)
    axes[1].plot(x, base_df['gini_proxy'],  color=color, lw=2, linestyle='--')
    axes[1].plot(x, decay_df['gini_proxy'], color=color, lw=2, label=label+' (decay)')
    axes[2].plot(x, base_df['fiscal_space_bn']/1000,  color=color, lw=2, linestyle='--')
    axes[2].plot(x, decay_df['fiscal_space_bn']/1000, color=color, lw=2, label=label+' (decay)')

axes[0].set_title('Governance Quality G(t)', fontweight='bold'); axes[0].set_ylabel('G (1=full capacity, 0.1=floor)')
axes[1].set_title('Gini: static (--) vs decay (—)', fontweight='bold'); axes[1].set_ylabel('Gini proxy')
axes[2].set_title('Fiscal Space $T: static (--) vs decay (—)', fontweight='bold'); axes[2].set_ylabel('$T')
axes[2].axhline(0, color='k', lw=0.8, linestyle=':')
for ax in axes:
    ax.grid(True, alpha=0.25); ax.set_xlabel('Year'); ax.legend(fontsize=7)
plt.tight_layout(); plt.show()


## MPC Sensitivity

The ownership-beats-redistribution finding rests primarily on the MPC differential. This cell sweeps concentrated capital MPC from 0.10 to 0.45 to show how sensitive the result is.


In [ ]:
## MPC Sensitivity Analysis
# The ownership-beats-redistribution finding is driven by the MPC differential
# between concentrated capital (~0.25) and distributed/yeoman income (~0.78).
# This cell sweeps the concentrated MPC to show how sensitive the result is.

# Temporarily override MPC_BASE_TOP1 and MPC_BASE_NEXT9
import copy

mpc_sweeps = [0.10, 0.20, 0.25, 0.35, 0.45]  # concentrated capital MPC
base_mpc   = MPC_BASE_TOP1  # save original

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
fig.suptitle('Sensitivity: Concentrated Capital MPC (baseline = 0.12)', fontweight='bold')

colors_sweep = ['#d62728','#ff7f0e','#1f77b4','#2ca02c','#9467bd']

for mpc_val, col in zip(mpc_sweeps, colors_sweep):
    MPC_BASE_TOP1  = mpc_val
    MPC_BASE_NEXT9 = min(mpc_val + 0.18, 0.50)  # maintain relative spread
    df_conc = run(mid=5, yeomen=0.00, tax_k=0.50, enforcement=1.0)
    df_yeo  = run(mid=5, yeomen=0.60, tax_k=0.50, enforcement=1.0)
    axes[0].plot(df_conc['year'], df_conc['gini_proxy'], color=col, lw=2, linestyle='--')
    axes[0].plot(df_yeo['year'],  df_yeo['gini_proxy'],  color=col, lw=2,
                 label=f'conc MPC={mpc_val:.2f}')
    axes[1].plot(df_conc['year'], df_conc['effective_welfare_k'], color=col, lw=2, linestyle='--')
    axes[1].plot(df_yeo['year'],  df_yeo['effective_welfare_k'],  color=col, lw=2,
                 label=f'conc MPC={mpc_val:.2f}')

MPC_BASE_TOP1  = base_mpc  # restore
MPC_BASE_NEXT9 = 0.30

axes[0].set_title('Gini: concentrated (--) vs high-yeomen (—)', fontweight='bold')
axes[0].set_ylabel('Gini proxy'); axes[0].set_xlabel('Year')
axes[1].set_title('Welfare $k/yr: concentrated (--) vs high-yeomen (—)', fontweight='bold')
axes[1].set_ylabel('$k/yr'); axes[1].set_xlabel('Year')
for ax in axes:
    ax.grid(True, alpha=0.25); ax.legend(fontsize=7)
plt.tight_layout(); plt.show()
print('Takeaway: the ownership advantage persists across the MPC sweep,'
      ' but narrows as concentrated MPC rises. At MPC_conc > 0.45 the gap closes substantially.')


## Interactive Scenario Explorer

Use the sliders to define your own scenario and compare it against the paper's key cases in real time.

*If widgets don't render (e.g. on GitHub), scroll down to the static version and edit `MY_SCENARIO` directly.*

In [ ]:
# ── Try to load widgets; fall back gracefully if unavailable ─────────────────
try:
    import ipywidgets as widgets
    from IPython.display import display, clear_output
    HAS_WIDGETS = True
except ImportError:
    HAS_WIDGETS = False
    print("ipywidgets not available — use the MY_SCENARIO dict below instead.")

if HAS_WIDGETS:
    out = widgets.Output()

    w_mid         = widgets.IntSlider(value=5,    min=5,    max=14,  step=1,    description="Automation speed (mid)", style={"description_width":"200px"}, layout=widgets.Layout(width="700px"))
    w_yeomen      = widgets.FloatSlider(value=0.30, min=0.0,  max=0.60, step=0.05, description="Yeomen fraction",        style={"description_width":"200px"}, layout=widgets.Layout(width="700px"))
    w_tax_k       = widgets.FloatSlider(value=0.35, min=0.20, max=0.50, step=0.05, description="Capital tax rate",       style={"description_width":"200px"}, layout=widgets.Layout(width="700px"))
    w_enforcement = widgets.FloatSlider(value=1.00, min=0.30, max=1.00, step=0.05, description="Enforcement",            style={"description_width":"200px"}, layout=widgets.Layout(width="700px"))
    w_dao_frac    = widgets.FloatSlider(value=0.10, min=0.0,  max=0.25, step=0.05, description="DAO fraction",           style={"description_width":"200px"}, layout=widgets.Layout(width="700px"))
    w_levy        = widgets.FloatSlider(value=0.0,  min=0.0,  max=1.00, step=0.10, description="Levy progressivity",     style={"description_width":"200px"}, layout=widgets.Layout(width="700px"))
    w_public_ai   = widgets.FloatSlider(value=0.0,  min=0.0,  max=0.90, step=0.10, description="Public AI fraction",     style={"description_width":"200px"}, layout=widgets.Layout(width="700px"))
    w_decay       = widgets.Checkbox(value=False, description="Enable governance decay", style={"description_width":"200px"})

    speed_label = widgets.HTML("<b>← faster automation &nbsp;&nbsp;&nbsp; slower →</b>")

    COMPARE_AGAINST = [
        "Fast / No yeomen / Low tax",
        "Fast / High yeomen / High tax",
        "Fast / Public AI 90%",
    ]
    COMPARE_COLORS = ["#d62728", "#2ca02c", "#9467bd"]

    def update(*args):
        with out:
            clear_output(wait=True)
            my_df = run(
                mid=w_mid.value, yeomen=w_yeomen.value, tax_k=w_tax_k.value,
                dao_frac=w_dao_frac.value, enforcement=w_enforcement.value,
                levy_prog=w_levy.value, public_ai_frac=w_public_ai.value,
                enable_decay=w_decay.value,
            )
            fig, axes = plt.subplots(2, 3, figsize=(18, 9))
            fig.suptitle("Your scenario vs. paper benchmarks", fontsize=13, fontweight="bold")

            plot_cols = [
                ("gini_proxy",          "Gini proxy",             1,      "Inequality"),
                ("effective_welfare_k", "Welfare floor ($k/yr)",  1,      "Welfare"),
                ("gdp_nom_bn",          "Nominal GDP ($T)",        1/1000, "Nominal GDP"),
                ("redistributable_bn",  "Redistributable ($T)",   1/1000, "Redistribution capacity"),
                ("labor_share_pct",     "Labour share (%)",        1,      "Labour share"),
                ("debt_gdp_ratio",      "Debt / GDP",              1,      "Sovereign debt"),
            ]

            for ax, (col, ylabel, scale, title) in zip(axes.flat, plot_cols):
                for name, color in zip(COMPARE_AGAINST, COMPARE_COLORS):
                    ax.plot(results[name]["year"], results[name][col] * scale,
                            color=color, lw=1.5, linestyle="--", alpha=0.7, label=name.replace("Fast / ", ""))
                ax.plot(my_df["year"], my_df[col] * scale,
                        color="#000000", lw=2.5, label="Your scenario")
                ax.set_title(title, fontweight="bold"); ax.set_ylabel(ylabel, fontsize=9)
                ax.set_xlabel("Year", fontsize=9); ax.grid(True, alpha=0.25)
                ax.legend(fontsize=7)

            plt.tight_layout(); plt.show()

            # Summary table
            print(f"\n{'Metric':<38} {'t+5':>8} {'t+10':>8} {'t+20':>8}")
            print("-" * 64)
            for col, label, fmt in [
                ("gini_proxy",          "Gini proxy",              "{:.3f}"),
                ("effective_welfare_k", "Welfare floor ($k/yr)",   "${:.0f}k"),
                ("gdp_nom_bn",          "Nominal GDP ($T)",         lambda v: f"${v/1000:.1f}T"),
                ("labor_share_pct",     "Labour share (%)",         "{:.0f}%"),
                ("debt_gdp_ratio",      "Debt/GDP",                 "{:.2f}x"),
                ("fiscal_space_bn",     "Fiscal space ($T)",        lambda v: f"${v/1000:.1f}T"),
                ("compute_dividend_k",  "Compute dividend ($k/yr)", "${:.1f}k"),
            ]:
                row = f"{label:<38}"
                for yr in [5, 10, 20]:
                    v = my_df[my_df.t == yr][col].values[0]
                    row += f"{(fmt(v) if callable(fmt) else fmt.format(v)):>8}"
                print(row)

    for w in [w_mid, w_yeomen, w_tax_k, w_enforcement, w_dao_frac, w_levy, w_public_ai, w_decay]:
        w.observe(update, names="value")

    col_left = widgets.VBox([
        widgets.HTML("<b>Automation</b>"),
        w_mid, speed_label,
        widgets.HTML("<br><b>Ownership structure</b>"),
        w_yeomen, w_dao_frac, w_public_ai,
    ])
    col_right = widgets.VBox([
        widgets.HTML("<b>Tax &amp; enforcement</b>"),
        w_tax_k, w_enforcement, w_levy,
        widgets.HTML("<br><b>Dynamics</b>"),
        w_decay,
    ])
    controls = widgets.HBox([col_left, col_right],
                            layout=widgets.Layout(justify_content="space-around", width="100%"))
    display(widgets.VBox([controls, out]))
    update()  # render immediately

# ── Static fallback (also useful for non-interactive export) ─────────────────
else:
    MY_SCENARIO = dict(
        mid=5, yeomen=0.30, tax_k=0.35, dao_frac=0.10,
        levy_prog=0.5, dao_govt_rate=0.0, public_ai_frac=0.0, enforcement=1.0,
    )
    my_df = run(**MY_SCENARIO)
    print("Gini t+10:", round(my_df[my_df.t==10]["gini_proxy"].values[0], 3))
    print("Welfare t+10:", round(my_df[my_df.t==10]["effective_welfare_k"].values[0], 1), "$k/yr")


## Explore Any Variable

Print all columns available in the DataFrame, then inspect any one in detail.

In [ ]:
# All available columns
sample_df = results['Fast / No yeomen / Low tax']
print('Available columns:')
for col in sample_df.columns:
    print(f'  {col}')

In [ ]:
# Plot any column across all highlighted scenarios
COLUMN_TO_PLOT = 'debt_gdp_ratio'   # ← change this
SCALE          = 1                   # ← change if needed (e.g. 1/1000 for $T)

fig, ax = plt.subplots(figsize=(12, 5))
for i, (name, df) in enumerate(results.items()):
    if name in HIGHLIGHT:
        ax.plot(df['year'], df[COLUMN_TO_PLOT] * SCALE, label=name,
                color=COLORS[i % len(COLORS)], lw=2)
ax.set_title(COLUMN_TO_PLOT, fontweight='bold')
ax.set_xlabel('Year')
ax.grid(True, alpha=0.3)
ax.legend(fontsize=8)
plt.tight_layout()
plt.show()

## LLM Household Agents — Optional Extension

The companion script `model_sfc_llm.py` replaces the parametric model's fixed consumption rules with live Gemini Flash calls. Each of the six household archetypes receives a structured prompt describing their personal financial situation and the macro environment each simulated year; Gemini returns a JSON spending/saving decision with reasoning.

**What this shows:**
- Whether LLM-simulated households qualitatively behave like the archetypes assumed in the model
- Whether the MPC differential (concentrated capital ~25%, workers ~78%) emerges from the LLM decisions or requires it to be imposed

**What it doesn't show:**
- Independent validation — LLMs are trained on economics literature and tend to reproduce textbook-rational behaviour when prompted. The MPC patterns may simply reflect their training.
- Aggregate macro divergence — the six-household SFC model is too small for micro heterogeneity to shift aggregate outcomes substantially.

The cell below loads reasoning from a previous 210-call run (cached in `.llm_cache/`). The live run cell requires a `GEMINI_API_KEY` environment variable.

In [ ]:
import json, glob, statistics
from pathlib import Path

CACHE_DIR = Path(".llm_cache")
files = sorted(CACHE_DIR.glob("*.json")) if CACHE_DIR.exists() else []
all_decisions = [json.loads(f.read_text()) for f in files]
decisions_with_reasoning = [d for d in all_decisions if d.get("reasoning", "").strip()]

print(f"{len(decisions_with_reasoning)} cached LLM decisions found")

if not decisions_with_reasoning:
    print("""
No cached decisions available in this environment.
The cache is generated locally when model_sfc_llm.py is run with a valid GEMINI_API_KEY.

To generate it:
  1. Clone the repository locally
  2. Set GEMINI_API_KEY in your environment
  3. Run: python3 model_sfc_llm.py "Fast / High yeomen / High tax"
  4. Re-upload the .llm_cache/ directory to Colab, or run the notebook locally

The cached reasoning snippets from a previous run are reproduced below for illustration:
""")
    SAMPLE_REASONING = [
        {"persona": "Concentrated capital owner",
         "consume_k": 221, "human_svc_share": 0.15,
         "reasoning": "As a concentrated capital owner, I prioritize reinvesting a substantial portion of my stable income into AI/robotic assets. High interest payments consume nearly half my income, necessitating a reduction in consumption spending."},
        {"persona": "Yeomen enterprise operator",
         "consume_k": 49, "human_svc_share": 0.20,
         "reasoning": "Maintaining my established consumption fraction, supported by stable income and UBI, allows for continued debt reduction and steady growth in net wealth, aligning with my cautious approach to finances."},
        {"persona": "DAO / commons contributor",
         "consume_k": 32, "human_svc_share": 0.45,
         "reasoning": "With continued growth in my DAO income, I can comfortably increase my spending, prioritizing human services and experiences, while reinvesting some back into commons projects."},
        {"persona": "Displaced worker on UBI",
         "consume_k": 18, "human_svc_share": 0.10,
         "reasoning": "With stable UBI and a strong emergency fund, I will prioritize aggressive debt reduction to alleviate anxiety, while maintaining essential living standards and supporting my retraining efforts."},
        {"persona": "Compute dividend recipient",
         "consume_k": 28, "human_svc_share": 0.30,
         "reasoning": "With the substantial increase in UBI to $30.6k, I feel very secure. I will maintain my increased savings rate, while still enjoying a significant boost in spending on experiences and small luxuries."},
        {"persona": "Human economy worker",
         "consume_k": 55, "human_svc_share": 0.35,
         "reasoning": "My income is stable in a human-centric sector, allowing me to save a small portion after covering my expenses, including rising housing costs."},
    ]
    print("=" * 72)
    for d in SAMPLE_REASONING:
        print(f"\n{d['persona']}")
        print(f"  consume=${d['consume_k']}k/yr  human-svc share={d['human_svc_share']:.0%}")
        print(f"  \"{d['reasoning']}\"")
    print("\n" + "=" * 72)
    print("\nNote: Capital owner narratives emphasise reinvestment; displaced workers")
    print("and dividend recipients spend near their full income — consistent with")
    print("the MPC differential assumed in the parametric model.")

else:
    # Full analysis from real cache
    persona_keywords = {
        "Concentrated capital owner": ["reinvest", "capital", "dividends", "portfolio", "advisor"],
        "Yeomen enterprise operator": ["business", "costs", "revenue", "platform", "borrow to grow"],
        "DAO / commons contributor":  ["commons", "cooperative", "community", "DAO"],
        "Displaced worker on UBI":    ["UBI", "retrain", "anxious", "essentials", "job"],
        "Compute dividend recipient":  ["dividend", "windfall", "unconditional", "luxuries"],
        "Human economy worker":        ["care", "trades", "wage", "housing", "cost of living"],
    }
    def infer_persona(reasoning):
        r = reasoning.lower()
        scores = {n: sum(1 for k in kws if k.lower() in r) for n, kws in persona_keywords.items()}
        best = max(scores, key=scores.get)
        return best if scores[best] > 0 else "Unknown"

    from collections import defaultdict
    by_persona = defaultdict(list)
    for d in decisions_with_reasoning:
        by_persona[infer_persona(d["reasoning"])].append(d)

    print("=" * 72)
    for persona, decisions in sorted(by_persona.items()):
        avg_consume = statistics.mean(d["consume_k"] for d in decisions)
        avg_human   = statistics.mean(d.get("human_svc_share", 0) for d in decisions)
        print(f"\n{persona}")
        print(f"  n={len(decisions)}, avg consume=${avg_consume:.0f}k/yr, avg human-svc share={avg_human:.0%}")
        for d in decisions[:2]:
            print(f"  \"{d['reasoning'][:130]}\"")
    print("\n" + "=" * 72)

    consumes_all = [d["consume_k"] for d in decisions_with_reasoning]
    s = sorted(consumes_all)
    print(f"\nConsumption distribution: median=${statistics.median(s):.0f}k  "
          f"mean=${statistics.mean(s):.0f}k  "
          f"p10=${s[len(s)//10]:.0f}k  p90=${s[9*len(s)//10]:.0f}k")


In [ ]:
# ── Live LLM run (requires GEMINI_API_KEY) ────────────────────────────────────
# Uncomment and set your API key to re-run with fresh LLM decisions.
# Each run makes ~210 Gemini Flash calls (~$0.01 at current pricing).
#
# import os, subprocess
# os.environ["GEMINI_API_KEY"] = "your_key_here"   # or set in terminal before launching notebook
#
# result = subprocess.run(
#     ["python3", "model_sfc_llm.py", "Fast / High yeomen / High tax"],
#     capture_output=True, text=True
# )
# print(result.stdout[-3000:])   # tail of output — full comparison table + chart path
#
# The chart is saved to ai_economy_sfc_llm.png — view it with:
# from IPython.display import Image
# Image("ai_economy_sfc_llm.png")

# ── Show result from previous run ────────────────────────────────────────────
from IPython.display import Image, display
import os
if os.path.exists("ai_economy_sfc_llm.png"):
    print("Chart from previous LLM run (Fast / High yeomen / High tax):")
    display(Image("ai_economy_sfc_llm.png", width=900))
else:
    print("No previous LLM run chart found. Run the cell above to generate one.")
